# EDA v2 — Nova Configuração de Variáveis

## Diferenças em relação ao EDA v1

| Item | EDA v1 | EDA v2 |
|---|---|---|
| Filtro | Todos os status | **Apenas `delivered`** |
| Outcome | `review_positivo` (score ≥ 4) | **`score_review` (score ∈ {3,4,5} = 1)** |
| Features | Subconjunto | **Conjunto expandido** |

## Outcome
```
score_review = 1  se review_score ∈ {3, 4, 5}  (neutro a positivo)
score_review = 0  se review_score ∈ {1, 2}      (negativo)
```

## Features analisadas
| Domínio | Variáveis |
|---|---|
| **Geográfico** | `customer_city`, `customer_state` |
| **Financeiro** | `total_price`, `avg_price`, `total_freight`, `avg_freight`, `total_payment`, `avg_payment`, `installment_value` |
| **Pedido** | `n_items`, `n_item_distinct_categ`, `n_items_missing_info`, `max_installments`, `n_payments_type` |
| **Produto** | `avg_weight`, `avg_length`, `avg_height`, `avg_width` |
| **Temporal** | `purchase_weekday`, `purchase_month` |
| **Logístico** | `is_delayed` (criada neste notebook) |

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from app.config.settings import INTERIM_DATA_DIR

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Carregamento e filtro

In [ ]:
df_raw = pd.read_parquet(os.path.join(INTERIM_DATA_DIR, "interim_dataset.parquet"))

print(f"Dataset completo: {df_raw.shape[0]:,} pedidos × {df_raw.shape[1]} colunas")
print(f"\nDistribuição order_status:")
print(df_raw['order_status'].value_counts())

# Filtro: apenas pedidos entregues
df = df_raw[df_raw['order_status'] == 'delivered'].copy()
print(f"\nApós filtro (delivered): {df.shape[0]:,} pedidos ({df.shape[0]/df_raw.shape[0]*100:.1f}% do total)")

## 3. Criação das variáveis

In [ ]:
# --- Datas ---
for col in ['order_purchase_timestamp', 'order_delivered_customer_date',
            'order_estimated_delivery_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# --- Outcome: score_review ---
# 1 = neutro a positivo (score 3, 4 ou 5)
# 0 = negativo (score 1 ou 2)
df['score_review'] = df['review_score'].apply(
    lambda x: 1 if x in [3, 4, 5] else (0 if x in [1, 2] else np.nan)
).astype(float)

# --- Logístico ---
df['delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

df['is_delayed']         = (df['delay_days'] > 0).astype(int)       # qualquer atraso
df['atraso_leve']        = ((df['delay_days'] > 0) & (df['delay_days'] <= 7)).astype(int)  # 1-7 dias
df['atraso_grave']       = (df['delay_days'] > 7).astype(int)        # mais de 7 dias
df['entrega_antecipada'] = (df['delay_days'] < -7).astype(int)       # 7+ dias antes do prazo

# --- Produto ---
df['volume_cm3']      = df['avg_length'] * df['avg_height'] * df['avg_width']
df['produto_pesado']  = (df['avg_weight'] > 1000).astype(int)        # acima de 1 kg
df['produto_grande']  = (df['volume_cm3'] > df['volume_cm3'].median()).astype(int)

# --- Financeiro ---
df['frete_desproporcional'] = (
    (df['avg_freight'] / df['avg_price'].replace(0, np.nan)) > 0.30
).astype(float)

p75_price = df['total_price'].quantile(0.75)
df['pedido_alto_valor'] = (df['total_price'] > p75_price).astype(int)
df['alto_parcelamento'] = (df['max_installments'] > 6).astype(int)

# --- Resumo ---
novas = {
    'is_delayed':             df['is_delayed'],
    'atraso_leve (1-7d)':     df['atraso_leve'],
    'atraso_grave (>7d)':     df['atraso_grave'],
    'entrega_antecipada':     df['entrega_antecipada'],
    'produto_pesado':         df['produto_pesado'],
    'produto_grande':         df['produto_grande'],
    'frete_desproporcional':  df['frete_desproporcional'],
    'pedido_alto_valor':      df['pedido_alto_valor'],
    'alto_parcelamento':      df['alto_parcelamento'],
}

print("Variáveis criadas — prevalência:")
print(f"{'Variável':<30} {'Prev%':>6}  {'N':>7}")
print("-" * 48)
for nome, serie in novas.items():
    s = serie.dropna()
    print(f"{nome:<30} {s.mean()*100:>5.1f}%  {int(s.sum()):>7,}")

print(f"\nOutcome score_review:")
print(f"  1 (neutro/positivo): {df['score_review'].sum():,.0f} ({df['score_review'].mean()*100:.1f}%)")
print(f"  0 (negativo):        {(df['score_review']==0).sum():,} ({(df['score_review']==0).mean()*100:.1f}%)")
print(f"  Nulos:               {df['score_review'].isna().sum():,}")
print(f"\n  p75 total_price: R$ {p75_price:.2f}")

# Verificação: is_delayed = atraso_leve + atraso_grave
check = df['is_delayed'].sum() == df['atraso_leve'].sum() + df['atraso_grave'].sum()
print(f"\n  ✅ Consistência: is_delayed = atraso_leve + atraso_grave → {check}")

## 4. Seleção das features e visão geral

In [ ]:
FEATURES = [
    # Geográfico
    'customer_city', 'customer_state',
    # Financeiro
    'total_price', 'avg_price', 'total_freight', 'avg_freight',
    'total_payment', 'avg_payment', 'installment_value',
    # Pedido
    'n_items', 'n_item_distinct_categ', 'n_items_missing_info',
    'max_installments', 'n_payments_type',
    # Produto
    'avg_weight', 'avg_length', 'avg_height', 'avg_width', 'volume_cm3',
    # Temporal
    'purchase_weekday', 'purchase_month',
    # Logístico — binárias de atraso
    'is_delayed', 'atraso_leve', 'atraso_grave', 'entrega_antecipada',
    # Produto — binárias
    'produto_pesado', 'produto_grande',
    # Financeiro — binárias
    'frete_desproporcional', 'pedido_alto_valor', 'alto_parcelamento',
]

OUTCOME = 'score_review'

df_model = df[FEATURES + [OUTCOME]].copy()

print(f"Shape do dataset modelagem: {df_model.shape}")
print(f"\nNulos por variável (apenas variáveis com nulos):")
nulos = df_model.isnull().sum()
nulos_pct = (nulos / len(df_model) * 100).round(2)
resumo_nulos = pd.DataFrame({'N nulos': nulos, '% nulos': nulos_pct})
print(resumo_nulos[resumo_nulos['N nulos'] > 0].to_string())

## 5. Distribuição do outcome

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribuição do review_score original
counts = df['review_score'].value_counts().sort_index().dropna()
cores  = ['#d32f2f', '#e57373', '#ffd54f', '#81c784', '#388e3c']
bars   = axes[0].bar(counts.index, counts.values, color=cores, edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}\n({val/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=8)
axes[0].set_title('Distribuição original — review_score (1-5)\n(apenas delivered)', fontsize=11)
axes[0].set_xlabel('review_score')
axes[0].set_ylabel('Frequência')
axes[0].set_xticks([1, 2, 3, 4, 5])

# score_review binarizado
sc = df['score_review'].value_counts().sort_index()
cores2 = ['#d32f2f', '#388e3c']
labels2 = ['0 — Negativo\n(score 1 ou 2)', '1 — Neutro/Positivo\n(score 3, 4 ou 5)']
bars2 = axes[1].bar(labels2, sc.values, color=cores2, edgecolor='white', width=0.5)
for bar, val in zip(bars2, sc.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}\n({val/sc.sum()*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10)
axes[1].set_title('score_review binarizado\n(outcome do modelo causal)', fontsize=11)
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.savefig('../../reports/figures/eda_v2_outcome.png', dpi=120)
plt.show()

## 6. Estatísticas descritivas por domínio

In [ ]:
dominios = {
    'Financeiro':  ['total_price', 'avg_price', 'total_freight',
                    'avg_freight', 'total_payment', 'avg_payment', 'installment_value'],
    'Pedido':      ['n_items', 'n_item_distinct_categ', 'n_items_missing_info',
                    'max_installments', 'n_payments_type'],
    'Produto':     ['avg_weight', 'avg_length', 'avg_height', 'avg_width'],
    'Logístico':   ['delay_days', 'is_delayed'],
}

rows = []
for dominio, cols in dominios.items():
    for col in cols:
        if col not in df.columns:
            continue
        s = df[col].dropna()
        rows.append({
            'Domínio':  dominio,
            'Variável': col,
            'N':        f'{len(s):,}',
            'Média':    round(s.mean(), 2),
            'DP':       round(s.std(), 2),
            'Mediana':  round(s.median(), 2),
            'IQR':      round(s.quantile(0.75) - s.quantile(0.25), 2),
            'Mín':      round(s.min(), 2),
            'Máx':      round(s.max(), 2),
        })

df_desc = pd.DataFrame(rows).set_index(['Domínio', 'Variável'])
print(df_desc.to_string())

## 7. Distribuição das variáveis numéricas contínuas

In [ ]:
numericas = [
    'total_price', 'avg_price', 'total_freight', 'avg_freight',
    'total_payment', 'installment_value',
    'n_items', 'n_item_distinct_categ', 'avg_weight',
    'avg_length', 'avg_height', 'avg_width',
    'max_installments', 'n_payments_type'
]

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(numericas):
    data = df[col].dropna()
    p99  = data.quantile(0.99)
    data_plot = data[data <= p99]
    axes[i].hist(data_plot, bins=40, color='steelblue', alpha=0.75, edgecolor='white')
    axes[i].set_title(f'{col}\n(p99={p99:.1f})', fontsize=8)
    axes[i].tick_params(labelsize=7)

for j in range(len(numericas), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das variáveis numéricas — pedidos delivered (até p99)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../../reports/figures/eda_v2_numericas.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Variáveis categóricas e temporais

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# Top 10 estados
df['customer_state'].value_counts().head(10).plot(
    kind='bar', ax=axes[0,0], color='steelblue', alpha=0.8)
axes[0,0].set_title('Top 10 estados por volume de pedidos')
axes[0,0].set_xlabel('Estado')
axes[0,0].tick_params(axis='x', rotation=0)

# Top 10 cidades
df['customer_city'].value_counts().head(10).plot(
    kind='bar', ax=axes[0,1], color='teal', alpha=0.8)
axes[0,1].set_title('Top 10 cidades por volume de pedidos')
axes[0,1].set_xlabel('Cidade')
axes[0,1].tick_params(axis='x', rotation=30)

# Dia da semana
dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
df['purchase_weekday'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1,0], color='coral', alpha=0.8)
axes[1,0].set_title('Compras por dia da semana')
axes[1,0].set_xticklabels(dias, rotation=0)

# Mês
df['purchase_month'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1,1], color='coral', alpha=0.8)
axes[1,1].set_title('Compras por mês')
axes[1,1].set_xlabel('Mês')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../../reports/figures/eda_v2_categoricas.png', dpi=120)
plt.show()

## 9. Associação das features com o outcome (score_review)

In [ ]:
# Taxa de score_review=1 por grupo (variáveis categóricas/binárias)
cats = ['is_delayed', 'purchase_weekday', 'purchase_month', 'customer_state']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(cats):
    taxa = df.groupby(col)['score_review'].mean().sort_values()
    if len(taxa) > 15:
        taxa = taxa.tail(15)  # top 15 para legibilidade
    taxa.plot(kind='barh', ax=axes[i], color='steelblue', alpha=0.8)
    axes[i].axvline(df['score_review'].mean(), color='red',
                    linestyle='--', linewidth=1.5, label='Média geral')
    axes[i].set_title(f'Taxa score_review=1 por {col}', fontsize=10)
    axes[i].set_xlabel('Proporção score_review=1')
    axes[i].legend(fontsize=8)

plt.suptitle('Associação bruta: features categóricas vs. score_review', fontsize=12)
plt.tight_layout()
plt.savefig('../../reports/figures/eda_v2_assoc_cats.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Diferença de médias: features numéricas entre score_review=0 e score_review=1
num_cols = [
    'total_price', 'avg_price', 'total_freight', 'avg_freight',
    'n_items', 'n_item_distinct_categ', 'avg_weight',
    'max_installments', 'n_payments_type', 'delay_days'
]

rows = []
for col in num_cols:
    g0 = df[df['score_review'] == 0][col].dropna()
    g1 = df[df['score_review'] == 1][col].dropna()
    diff = g1.mean() - g0.mean()
    rows.append({
        'Feature':          col,
        'Média score=0':    round(g0.mean(), 3),
        'Média score=1':    round(g1.mean(), 3),
        'Diferença (1-0)':  round(diff, 3),
        'Direção':          '↑ score=1 maior' if diff > 0 else '↓ score=0 maior'
    })

df_assoc = pd.DataFrame(rows).sort_values('Diferença (1-0)', key=abs, ascending=False)
print("Diferença de médias entre grupos de score_review:")
print(df_assoc.to_string(index=False))

## 10. Correlação entre features numéricas e outcome

In [ ]:
num_features = [
    'total_price', 'avg_price', 'total_freight', 'avg_freight',
    'total_payment', 'avg_payment', 'installment_value',
    'n_items', 'n_item_distinct_categ', 'n_items_missing_info',
    'max_installments', 'n_payments_type',
    'avg_weight', 'avg_length', 'avg_height', 'avg_width',
    'is_delayed', 'score_review'
]

corr = df[num_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-0.6, vmax=0.6, ax=ax, mask=mask,
            linewidths=0.4, square=True, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 7})
ax.set_title('Correlação entre features e outcome (score_review)', fontsize=12, pad=15)
plt.tight_layout()
plt.savefig('../../reports/figures/eda_v2_correlacao.png', dpi=120)
plt.show()

# Correlação específica com o outcome
print("\nCorrelação de cada feature com score_review:")
corr_outcome = corr['score_review'].drop('score_review').sort_values(key=abs, ascending=False)
print(corr_outcome.round(4).to_string())

## 11. Análise de is_delayed — principal candidato a tratamento

In [ ]:
# Trabalha apenas com linhas que têm delay_days e score_review válidos
df_delay = df.dropna(subset=['delay_days', 'score_review']).copy()
df_delay['is_delayed'] = (df_delay['delay_days'] > 0).astype(int)

med_delay = df_delay['delay_days'].median()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Subplot 1: Distribuição de delay_days ---
delay_clip = df_delay['delay_days'].clip(-30, 30).values
axes[0].hist(delay_clip, bins=50, color='coral', alpha=0.75, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=2, linestyle='--', label='Prazo')
axes[0].axvline(med_delay, color='navy', linewidth=1.5,
                linestyle='-.', label=f'Mediana={med_delay:.0f}d')
axes[0].set_title('Distribuição de delay_days\n(clipado em ±30)', fontsize=10)
axes[0].set_xlabel('dias (positivo = atrasado)')
axes[0].legend(fontsize=8)

# --- Subplot 2: Taxa de score_review por is_delayed ---
taxa_delay = df_delay.groupby('is_delayed')['score_review'].mean()
cores_d = ['#388e3c', '#d32f2f']
bars = axes[1].bar(['No prazo\n(is_delayed=0)', 'Atrasado\n(is_delayed=1)'],
                   taxa_delay.values * 100, color=cores_d, alpha=0.8, width=0.5)
for bar, val in zip(bars, taxa_delay.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val*100:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[1].set_title('Taxa score_review=1\npor grupo de atraso', fontsize=10)
axes[1].set_ylabel('%  score_review = 1')
axes[1].set_ylim(0, 110)
diff_bruta = (taxa_delay[1] - taxa_delay[0]) * 100
axes[1].text(0.5, 0.52, f'Diferença bruta:\n{diff_bruta:+.1f} p.p.',
             ha='center', fontsize=11, color='navy',
             transform=axes[1].transAxes)

# --- Subplot 3: delay_days por grupo de score_review ---
for score, cor, label in [(0, '#d32f2f', 'score=0 (negativo)'),
                           (1, '#388e3c', 'score=1 (neutro/positivo)')]:
    data = df_delay[df_delay['score_review'] == score]['delay_days'].clip(-20, 30).values
    axes[2].hist(data, bins=40, alpha=0.5, color=cor, label=label, edgecolor='white')
axes[2].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[2].set_title('delay_days por score_review', fontsize=10)
axes[2].set_xlabel('dias de atraso')
axes[2].legend(fontsize=8)

fig.suptitle('Análise de is_delayed vs. score_review', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/eda_v2_is_delayed.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

print(f"\nNulos em delay_days removidos da análise: {df['delay_days'].isna().sum():,}")
print(f"\nPrevalência is_delayed (subset sem NaN):")
print(f"  Atrasados: {df_delay['is_delayed'].sum():,} ({df_delay['is_delayed'].mean()*100:.1f}%)")
print(f"  No prazo:  {(df_delay['is_delayed']==0).sum():,} ({(df_delay['is_delayed']==0).mean()*100:.1f}%)")
print(f"\nDiferença bruta score_review:")
print(f"  No prazo:  {taxa_delay[0]*100:.1f}%")
print(f"  Atrasado:  {taxa_delay[1]*100:.1f}%")
print(f"  Diferença: {diff_bruta:+.1f} p.p.")

## 12. Prevalência das features binárias/discretas candidatas a tratamento

In [ ]:
df_ana = df.dropna(subset=['score_review']).copy()

candidatos = {
    # Logístico — atraso
    'is_delayed (>0d)':       df_ana['is_delayed'],
    'atraso_leve (1-7d)':     df_ana['atraso_leve'],
    'atraso_grave (>7d)':     df_ana['atraso_grave'],
    'entrega_antecipada(<-7d)': df_ana['entrega_antecipada'],
    # Pedido
    'pedido_multi_item':      (df_ana['n_items'] > 1).astype(int),
    'pedido_multi_categ':     (df_ana['n_item_distinct_categ'] > 1).astype(int),
    # Produto
    'produto_pesado (>1kg)':  df_ana['produto_pesado'],
    'produto_grande (>med)':  df_ana['produto_grande'],
    # Financeiro
    'frete_desproporcional':  df_ana['frete_desproporcional'],
    'pedido_alto_valor (>p75)': df_ana['pedido_alto_valor'],
    'alto_parcelamento (>6x)': df_ana['alto_parcelamento'],
    'parcelado (>1x)':        (df_ana['max_installments'] > 1).astype(int),
}

rows = []
for label, serie in candidatos.items():
    s = serie.dropna()
    prev = s.mean()
    mask1 = s == 1
    mask0 = s == 0
    t1 = df_ana.loc[mask1.index[mask1], 'score_review'].mean()
    t0 = df_ana.loc[mask0.index[mask0], 'score_review'].mean()
    diff = (t1 - t0) * 100
    ok = '✅' if 0.05 <= prev <= 0.95 else '⚠️ baixa' if prev < 0.05 else '⚠️ alta'
    rows.append({
        'Variável':          label,
        'Prevalência':       f'{prev*100:.1f}%',
        'OK para IPTW?':     ok,
        'Score T=1 (%)':     f'{t1*100:.1f}%',
        'Score T=0 (%)':     f'{t0*100:.1f}%',
        'Diff bruta (p.p.)': round(diff, 1),
    })

df_prev = pd.DataFrame(rows).sort_values('Diff bruta (p.p.)')
print("Candidatos a tratamento — prevalência e associação bruta com score_review:")
print(df_prev.to_string(index=False))

# Visualização
fig, ax = plt.subplots(figsize=(11, 6))
cores = ['#d32f2f' if v < 0 else '#388e3c' for v in df_prev['Diff bruta (p.p.)']]
bars = ax.barh(df_prev['Variável'], df_prev['Diff bruta (p.p.)'],
               color=cores, alpha=0.8, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Diferença bruta em score_review (p.p.)  |  T=1 vs T=0')
ax.set_title('Associação bruta dos candidatos a tratamento com score_review\n(pedidos delivered)', fontsize=11)
for bar, val in zip(bars, df_prev['Diff bruta (p.p.)']):
    offset = 0.8 if val >= 0 else -0.8
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../../reports/figures/eda_v2_candidatos.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Novo objetivo do TCC
#
# "Estimar o efeito causal de fatores operacionais (atraso na entrega,
#  características do produto, complexidade do pedido) sobre a satisfação
#  do cliente em transações de e-commerce concluídas, utilizando IPTW."
#
# População: pedidos com order_status == 'delivered'
# Outcome:   score_review (1 = score ∈ {3,4,5};  0 = score ∈ {1,2})

## 13. Proposta de variáveis para o modelo causal v2

### Novo objetivo do TCC
> **"Estimar o efeito causal de fatores operacionais — atraso na entrega, características do produto e complexidade do pedido — sobre a satisfação do cliente em transações de e-commerce concluídas, utilizando IPTW."**

**Por que essa mudança em relação ao ORS?**
- ORS mistura pedidos com status diferentes, dificultando interpretação causal
- Filtrar apenas `delivered` elimina viés de seleção pelo status do pedido
- `score_review` binário é mais estável e interpretável que score bruto 1-5

---

### Tratamentos candidatos

| # | Tratamento | Definição | Domínio | Nota |
|---|---|---|---|---|
| 1 | `is_delayed` | delay_days > 0 | Logístico | Efeito total do atraso |
| 2 | `atraso_leve` | 0 < delay_days ≤ 7 | Logístico | Atraso tolerável |
| 3 | `atraso_grave` | delay_days > 7 | Logístico | Atraso severo |
| 4 | `entrega_antecipada` | delay_days < −7 | Logístico | Efeito positivo |
| 5 | `pedido_multi_item` | n_items > 1 | Pedido | Complexidade do pedido |
| 6 | `produto_pesado` | avg_weight > 1.000g | Produto | Via logística |
| 7 | `produto_grande` | volume > mediana | Produto | Via logística |
| 8 | `frete_desproporcional` | avg_freight/avg_price > 30% | Financeiro | Percepção de custo |

> **Nota:** `atraso_leve` e `atraso_grave` são mutuamente exclusivos e somam `is_delayed`. Usar os três permite comparar "efeito dose-resposta" do atraso.

---

### Outcome
| Variável | Definição | Base |
|---|---|---|
| `score_review` | 1 se review_score ∈ {3,4,5} | Pedidos delivered |

---

### Confundidores por tratamento (sem colineridade estrutural)

| Tratamento | Confundidores | Excluídos |
|---|---|---|
| `is_delayed` / `atraso_leve` / `atraso_grave` | `customer_state`, `avg_price`, `avg_weight`, `n_items`, `purchase_month`, `purchase_weekday`, `n_items_missing_info` | `delay_days` |
| `entrega_antecipada` | `customer_state`, `avg_price`, `avg_weight`, `n_items`, `purchase_month` | `delay_days`, `is_delayed` |
| `produto_pesado` | `customer_state`, `avg_price`, `total_freight`, `n_items`, `purchase_month` | `avg_weight`, `volume_cm3` |
| `produto_grande` | `customer_state`, `avg_price`, `avg_weight`, `n_items`, `purchase_month` | `volume_cm3`, dimensões |
| `pedido_multi_item` | `customer_state`, `avg_price`, `avg_weight`, `purchase_month`, `purchase_weekday` | `n_items` |
| `frete_desproporcional` | `customer_state`, `avg_price`, `avg_weight`, `purchase_month`, `purchase_weekday` | `avg_freight`, `total_freight` |